In [1]:
# =================================================================================================
# Raw data : This is Questionnaire data from NHANES 2003-2004
# Purpose : Creating unified raw CSV file from NHANES questionnaire CSV files.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique respondent ID.
# Files Used : ALQ_C.csv, BPQ_C.csv, DIQ_C.csv, HIQ_C.csv, HUQ_C.csv, SMQ_C.csv, SMQFAM_C.csv, WHQ_C.csv
# Data Dictionary : CDC NHANES 2003-2004 Questionnaire Data Dictionary
# Data Availability Matrix : Created in this notebook using SEQN presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&CycleBeginYear=2003
# Developer Notes : The notebook was created for Windows desktop environment with Python3.
# File Creator : Neha Khairnar
# =================================================================================================
# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Paths below resolved for locally run datasets
# =================================================================================================

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data" / "2003_2004" / "Questionnnair"

alcohol_file = DATA_DIR / "ALQ_C_corrected.csv"
bp_chol_file = DATA_DIR / "BPQ_C_corrected.csv"
diabetes_file = DATA_DIR / "DIQ_C_corrected.csv"
insurance_file = DATA_DIR / "HIQ_C_corrected.csv"
access_file = DATA_DIR / "HUQ_C_corrected.csv"
smoking_file = DATA_DIR / "SMQ_C_corrected.csv"
household_smokers_file = DATA_DIR / "SMQFAM_C_corrected.csv"
weight_history_file = DATA_DIR / "WHQ_C_corrected.csv"
Housing_file = DATA_DIR / "HOQ_C_corrected.csv"
Occupation_file = DATA_DIR / "OCQ_C_corrected.csv"
medical_cndition_file = DATA_DIR / "MCQ_C_corrected.csv"
physical_file = DATA_DIR / "PAQ_C_corrected.csv"
Osteo_file = DATA_DIR / "OSQ_C_corrected.csv"

files = {
    "alcohol_ALQ_C": alcohol_file,
    "blood_pressure_cholesterol_BPQ_C": bp_chol_file,
    "diabetes_DIQ_C": diabetes_file,
    "health_insurance_HIQ_C": insurance_file,
    "hospital_access_HUQ_C": access_file,
    "smoking_SMQ_C": smoking_file,
    "household_smokers_SMQFAM_C": household_smokers_file,
    "weight_history_WHQ_C": weight_history_file,    
    "Housing_file_HOQ_C" : Housing_file,
    "Occupation_file_OCQ_C":Occupation_file,
    "medical_cndition_file_MCQ_C":medical_cndition_file,
    "physical_file" :physical_file,
    "Osteo_file" : Osteo_file
}

# =================================================================================================
# 📥 Step 3: Load ALL Files
# =================================================================================================

dfs = {}
for name, path in files.items():
    df_temp = pd.read_csv(path, dtype=str,keep_default_na=False)
    df_temp.columns = df_temp.columns.str.upper()
    dfs[name] = df_temp

summary = []
for name, df_temp in dfs.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
print("📊 NHANES Questionnaire File Structure Summary:\n")
display(summary_df[["file", "rows", "columns", "column_names"]])
for i, row in summary_df.iterrows():
    print("\n" + "="*80)
    print("FILE:", row["file"])
    print("COLUMNS:", len(row["column_names"]))
    print(row["column_names"])

# =================================================================================================
# 🔗 Step 5: Merge ALL columns using FULL OUTER JOIN on SEQN
# =================================================================================================

df = dfs["alcohol_ALQ_C"]
for name in [
    "blood_pressure_cholesterol_BPQ_C",
    "diabetes_DIQ_C",
    "health_insurance_HIQ_C",
    "hospital_access_HUQ_C",
    "smoking_SMQ_C",
    "household_smokers_SMQFAM_C",
    "weight_history_WHQ_C",    
    "Housing_file_HOQ_C",
    "Occupation_file_OCQ_C",
    "medical_cndition_file_MCQ_C",
    "physical_file",
    "Osteo_file",
]:
    df = df.merge(
        dfs[name],
        on="SEQN",
        how="outer",
        suffixes=("", f"_{name}")
    )

# =================================================================================================
# 💾 Step 6: Save Final Output
# =================================================================================================
print("📋 Final Columns After Merge + Rename:")
print(df.columns.tolist())

print("\n📊 Final Shape:", df.shape)
display(df.head(20))


OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_questionnaire_2003_2004_raw_corrected.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Saved to:")
print(OUTPUT_PATH)



📊 NHANES Questionnaire File Structure Summary:



,file,rows,columns,column_names
0,alcohol_ALQ_C,4742,9,"[SEQN, ALQ101, ALQ110, ALQ120Q, ALQ120U, ALQ13..."
1,blood_pressure_cholesterol_BPQ_C,6213,36,"[SEQN, BPQ010, BPQ020, BPQ030, BPQ040A, BPQ040..."
2,diabetes_DIQ_C,9645,17,"[SEQN, DIQ010, DID040G, DID040Q, DIQ050, DID06..."
3,health_insurance_HIQ_C,10122,10,"[SEQN, HID010, HID030A, HID030B, HID030C, HID0..."
4,hospital_access_HUQ_C,10122,16,"[SEQN, HUQ010, HUQ020, HUQ030, HUQ040, HUQ050,..."
5,smoking_SMQ_C,5041,42,"[SEQN, SMQ020, SMD030, SMQ040, SMQ050Q, SMQ050..."
6,household_smokers_SMQFAM_C,10122,9,"[SEQN, SMD410, SMD415, SMD415A, SMD415B, SMD41..."
7,weight_history_WHQ_C,6213,42,"[SEQN, WHD010, WHD020, WHQ030, WHQ040, WHD050,..."
8,Housing_file_HOQ_C,10122,17,"[SEQN, HOD011, HOD030, HOD040, HOD050, HOD060,..."
9,Occupation_file_OCQ_C,6213,25,"[SEQN, OCD150, OCQ180, OCQ210, OCD230, OCD240,..."



FILE: alcohol_ALQ_C
COLUMNS: 9
['SEQN', 'ALQ101', 'ALQ110', 'ALQ120Q', 'ALQ120U', 'ALQ130', 'ALQ140Q', 'ALQ140U', 'ALQ150']

FILE: blood_pressure_cholesterol_BPQ_C
COLUMNS: 36
['SEQN', 'BPQ010', 'BPQ020', 'BPQ030', 'BPQ040A', 'BPQ040B', 'BPQ040C', 'BPQ040D', 'BPQ040E', 'BPQ040F', 'BPQ043A', 'BPQ043B', 'BPQ043C', 'BPQ043D', 'BPQ050A', 'BPQ050B', 'BPQ050C', 'BPQ050D', 'BPQ050E', 'BPQ060', 'BPQ070', 'BPQ080', 'BPQ090A', 'BPQ090B', 'BPQ090C', 'BPQ090D', 'BPQ100A', 'BPQ100B', 'BPQ100C', 'BPQ100D', 'BPD110A', 'BPD110B', 'BPD110C', 'BPD120', 'BPD130', 'BPD140']

FILE: diabetes_DIQ_C
COLUMNS: 17
['SEQN', 'DIQ010', 'DID040G', 'DID040Q', 'DIQ050', 'DID060G', 'DID060Q', 'DIQ060U', 'DIQ070', 'DIQ080', 'DIQ090', 'DIQ100', 'DIQ110', 'DIQ120', 'DIQ130', 'DIQ140', 'DIQ150']

FILE: health_insurance_HIQ_C
COLUMNS: 10
['SEQN', 'HID010', 'HID030A', 'HID030B', 'HID030C', 'HID030D', 'HID030E', 'HID040', 'HIQ210', 'HIQ220']

FILE: hospital_access_HUQ_C
COLUMNS: 16
['SEQN', 'HUQ010', 'HUQ020', 'HUQ030', 'HUQ

,SEQN,ALQ101,ALQ110,ALQ120Q,ALQ120U,ALQ130,ALQ140Q,ALQ140U,ALQ150,BPQ010,...,OSQ040CG,OSD030CH,OSQ040CH,OSD030CI,OSQ040CI,OSD030CJ,OSQ040CJ,OSD050CJ,OSQ060,OSQ070
0,21005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,21007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21009,1,,0,,,,,2,2,...,,,,,,,,,2,
5,21010,1,,7,1,4,1,3,2,3,...,,,,,,,,,2,
6,21011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,21012,,,,,,,,,1,...,,,,,,,,,2,
8,21013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,21014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



✅ Saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_questionnaire_2003_2004_raw_corrected.csv
